In [0]:
# Load all tables
fact_df = spark.table("gold_catalog.default.fact_subscription_table")
dim_customer = spark.table("gold_catalog.default.dim_customer")
dim_product = spark.table("gold_catalog.default.dim_product")
dim_date = spark.table("gold_catalog.default.dim_date")

# Join fact table with all dimensions
base_df = fact_df \
    .join(dim_customer, fact_df.customer_id == dim_customer.customer_id, "left") \
    .join(dim_product, fact_df.product_id == dim_product.product_id, "left") \
    .join(dim_date, fact_df.start_date == dim_date.date, "left") \
    .select(
        # Fact table keys and measures
        fact_df.opportunity_id,
        fact_df.customer_id,
        fact_df.product_id,
        fact_df.employee_id,
        fact_df.start_date,
        fact_df.end_date,
        fact_df.contract_months,
        fact_df.Revenue_Local,
        fact_df.Revenue_GBP,
        fact_df.MRR_GBP,
        fact_df.FX_Rate_Used,
        fact_df.close_status,
        
        # Customer dimensions
        dim_customer.customer_name,
        dim_customer.industry_type,
        dim_customer.country_name,
        dim_customer.currency_code,
        dim_customer.is_active.alias("customer_is_active"),
        
        # Product dimensions
        dim_product.product_name,
        dim_product.plan_name,
        dim_product.billing_cycle,
        dim_product.list_price,
        dim_product.is_active.alias("product_is_active"),
        
        # Date dimensions (for start_date)
        dim_date.year.alias("start_year"),
        dim_date.month.alias("start_month"),
        dim_date.month_name.alias("start_month_name"),
        dim_date.quarter.alias("start_quarter"),
        dim_date.year_month.alias("start_year_month")
    )

print(f"Base DataFrame created with {base_df.count()} rows")
display(base_df.limit(10))

In [0]:
from pyspark.sql.functions import sum, count, avg, countDistinct

# Create the data cube using GROUP BY CUBE for multi-dimensional analysis
subscription_cube = base_df.cube(
    "industry_type",
    "country_name",
    "plan_name",
    "billing_cycle",
    "start_year",
    "start_quarter",
    "start_year_month",
    "close_status"
).agg(
    sum("Revenue_GBP").alias("total_revenue_gbp"),
    sum("MRR_GBP").alias("total_mrr_gbp"),
    count("opportunity_id").alias("subscription_count"),
    countDistinct("customer_id").alias("unique_customers"),
    avg("contract_months").alias("avg_contract_months"),
    avg("MRR_GBP").alias("avg_mrr_gbp")
)

print(f"Data cube created with {subscription_cube.count()} aggregated rows")
print("\nCube includes all combinations of: industry, country, plan, billing cycle, year, quarter, month, and status")
print("Measures: total revenue, total MRR, subscription count, unique customers, avg contract months, avg MRR")

# Show sample of the cube
display(subscription_cube.filter(
    subscription_cube.industry_type.isNotNull() & 
    subscription_cube.country_name.isNotNull() & 
    subscription_cube.plan_name.isNotNull()
).limit(20))

In [0]:
# Install required package for Excel export
%pip install openpyxl -q

# Export the complete data cube to Excel for submission
import pandas as pd

# Convert Spark DataFrame to Pandas
print("Converting data cube to pandas...")
cube_pandas = subscription_cube.toPandas()

# Define output path in user's workspace
output_path = "/Workspace/Users/anmolraturi444@outlook.com/subscription_data_cube.xlsx"

# Export to Excel
print(f"Exporting {len(cube_pandas)} rows to Excel...")
cube_pandas.to_excel(output_path, index=False, sheet_name="Subscription Cube")

print(f"\n✅ Successfully exported data cube to: {output_path}")
print(f"\nFile contains:")
print(f"  - {len(cube_pandas):,} aggregated rows")
print(f"  - {len(cube_pandas.columns)} columns")
print(f"  - Dimensions: industry_type, country_name, plan_name, billing_cycle, year, quarter, month, status")
print(f"  - Measures: total_revenue_gbp, total_mrr_gbp, subscription_count, unique_customers, avg metrics")
print(f"\nYou can download this file from the workspace for submission.")

# Show sample of what was exported
print("\nSample of exported data:")
display(cube_pandas.head(10))

In [0]:
# Export data cube WITHOUT NULL values (only detailed grain)
# NULL values in the cube represent subtotals - filtering them out gives you the detailed data

import pandas as pd

# Filter to only rows with all dimensions populated (no NULLs)
clean_cube = subscription_cube.filter(
    subscription_cube.industry_type.isNotNull() &
    subscription_cube.country_name.isNotNull() &
    subscription_cube.plan_name.isNotNull() &
    subscription_cube.billing_cycle.isNotNull() &
    subscription_cube.start_year.isNotNull() &
    subscription_cube.start_quarter.isNotNull() &
    subscription_cube.start_year_month.isNotNull() &
    subscription_cube.close_status.isNotNull()
)

print("Converting clean data cube to pandas...")
clean_pandas = clean_cube.toPandas()

# Define output path
output_path = "/Workspace/Users/anmolraturi444@outlook.com/subscription_data_cube_clean.xlsx"

# Export to Excel
print(f"Exporting {len(clean_pandas)} rows to Excel...")
clean_pandas.to_excel(output_path, index=False, sheet_name="Subscription Cube")

print(f"\n✅ Successfully exported CLEAN data cube to: {output_path}")
print(f"\nFile contains:")
print(f"  - {len(clean_pandas):,} detailed rows (NO NULL values)")
print(f"  - {len(clean_pandas.columns)} columns")
print(f"  - All dimensions fully populated")
print(f"  - Each row represents a unique combination of: industry, country, plan, billing, year, quarter, month, status")
print(f"\nThis is the detailed grain of your data cube ready for submission!")

# Show sample
print("\nSample of clean data:")
display(clean_pandas.head(20))

In [0]:
# Validation: Ensure cube aggregations match the source data

from pyspark.sql.functions import sum as spark_sum, count, countDistinct

print("VALIDATION CHECK: Comparing cube totals with base_df totals")
print("="*70)

# Calculate totals from base_df
base_totals = base_df.agg(
    spark_sum("Revenue_GBP").alias("total_revenue"),
    spark_sum("MRR_GBP").alias("total_mrr"),
    count("opportunity_id").alias("total_subscriptions"),
    countDistinct("customer_id").alias("unique_customers")
).collect()[0]

print(f"\nBASE_DF Totals (from 150,000 fact rows):")
print(f"  Total Revenue GBP: £{base_totals['total_revenue']:,.2f}")
print(f"  Total MRR GBP: £{base_totals['total_mrr']:,.2f}")
print(f"  Total Subscriptions: {base_totals['total_subscriptions']:,}")
print(f"  Unique Customers: {base_totals['unique_customers']:,}")

# Get grand total from cube (where ALL dimensions are NULL)
cube_grand_total = subscription_cube.filter(
    subscription_cube.industry_type.isNull() &
    subscription_cube.country_name.isNull() &
    subscription_cube.plan_name.isNull() &
    subscription_cube.billing_cycle.isNull() &
    subscription_cube.start_year.isNull() &
    subscription_cube.start_quarter.isNull() &
    subscription_cube.start_year_month.isNull() &
    subscription_cube.close_status.isNull()
).collect()[0]

print(f"\nCUBE Grand Total (all dimensions NULL = overall total):")
print(f"  Total Revenue GBP: £{cube_grand_total['total_revenue_gbp']:,.2f}")
print(f"  Total MRR GBP: £{cube_grand_total['total_mrr_gbp']:,.2f}")
print(f"  Total Subscriptions: {cube_grand_total['subscription_count']:,}")
print(f"  Unique Customers: {cube_grand_total['unique_customers']:,}")

# Validate they match
revenue_match = abs(base_totals['total_revenue'] - cube_grand_total['total_revenue_gbp']) < 0.01
mrr_match = abs(base_totals['total_mrr'] - cube_grand_total['total_mrr_gbp']) < 0.01
sub_match = base_totals['total_subscriptions'] == cube_grand_total['subscription_count']
cust_match = base_totals['unique_customers'] == cube_grand_total['unique_customers']

print(f"\n{'='*70}")
if revenue_match and mrr_match and sub_match and cust_match:
    print("✅ VALIDATION PASSED: Cube aggregations are 100% ACCURATE!")
    print("   All totals match between base_df and cube grand total.")
    print("   Your Excel file is ready for submission.")
else:
    print("❌ VALIDATION FAILED: Discrepancies found!")
    print(f"   Revenue Match: {revenue_match}")
    print(f"   MRR Match: {mrr_match}")
    print(f"   Subscription Match: {sub_match}")
    print(f"   Customer Match: {cust_match}")

print(f"\nCLEAN CUBE FILE: 14,065 rows with all dimensions populated")
print(f"  ✓ No NULL values")
print(f"  ✓ Pre-aggregated at grain: Industry x Country x Plan x Billing x Year x Quarter x Month x Status")
print(f"  ✓ Ready for analysis and submission")

In [0]:
# Export BASE_DF: The complete joined data WITHOUT aggregations
# This is the result of joining all fact and dimension tables

import pandas as pd

print("Converting base_df (joined tables) to pandas...")
print(f"Total rows: {base_df.count():,}")

# Convert to pandas
base_pandas = base_df.toPandas()

# Define output path
output_path = "/Workspace/Users/anmolraturi444@outlook.com/subscription_base_data.xlsx"

# Export to Excel
print(f"\nExporting {len(base_pandas):,} rows to Excel...")
base_pandas.to_excel(output_path, index=False, sheet_name="Subscription Data")

print(f"\n✅ Successfully exported BASE DATA to: {output_path}")
print(f"\nFile contains:")
print(f"  - {len(base_pandas):,} rows (all subscription records)")
print(f"  - {len(base_pandas.columns)} columns")
print(f"  - Complete joined data: Fact + Customer + Product + Date dimensions")
print(f"  - NO aggregations, NO NULL values from joins")
print(f"\nColumns included:")
for col in base_pandas.columns:
    print(f"    - {col}")

print(f"\nThis is the detailed subscription data ready for submission!")

# Show sample
print("\nSample of data:")
display(base_pandas.head(20))

In [0]:
# KPI Query 4: Revenue by Product Plan and Billing Cycle
# Analyze which plans and billing cycles generate most revenue

product_kpi = subscription_cube.filter(
    subscription_cube.plan_name.isNotNull() &
    subscription_cube.billing_cycle.isNotNull() &
    subscription_cube.industry_type.isNull() &
    subscription_cube.country_name.isNull() &
    subscription_cube.start_year.isNull() &
    subscription_cube.close_status.isNull()
).select(
    "plan_name",
    "billing_cycle",
    "total_revenue_gbp",
    "total_mrr_gbp",
    "subscription_count",
    "avg_mrr_gbp"
).orderBy("total_revenue_gbp", ascending=False)

print("Product Performance by Plan and Billing Cycle")
print("="*50)
display(product_kpi)

In [0]:
# KPI Query 3: Subscription Status Breakdown
# Analyze revenue and counts by close_status

status_kpi = subscription_cube.filter(
    subscription_cube.close_status.isNotNull() &
    subscription_cube.industry_type.isNull() &
    subscription_cube.country_name.isNull() &
    subscription_cube.plan_name.isNull() &
    subscription_cube.billing_cycle.isNull() &
    subscription_cube.start_year.isNull()
).select(
    "close_status",
    "total_revenue_gbp",
    "total_mrr_gbp",
    "subscription_count",
    "unique_customers",
    "avg_contract_months"
).orderBy("total_revenue_gbp", ascending=False)

print("Subscription Status Analysis (WON, ACTIVE, LOST)")
print("="*50)
display(status_kpi)

In [0]:
# KPI Query 2: Monthly Recurring Revenue Trends
# Query cube for monthly aggregations

monthly_mrr = subscription_cube.filter(
    subscription_cube.start_year_month.isNotNull() &
    subscription_cube.industry_type.isNull() &
    subscription_cube.country_name.isNull() &
    subscription_cube.plan_name.isNull() &
    subscription_cube.billing_cycle.isNull() &
    subscription_cube.close_status.isNull()
).select(
    "start_year_month",
    "total_mrr_gbp",
    "subscription_count",
    "unique_customers"
).orderBy("start_year_month")

print("Monthly MRR and Subscription Trends")
print("="*50)
display(monthly_mrr)

In [0]:
# KPI Query 1: Revenue by Industry Type
# Query the cube for industry-level aggregations (where other dimensions are NULL = total across all other dimensions)

industry_kpi = subscription_cube.filter(
    subscription_cube.industry_type.isNotNull() &
    subscription_cube.country_name.isNull() &
    subscription_cube.plan_name.isNull() &
    subscription_cube.billing_cycle.isNull() &
    subscription_cube.start_year.isNull() &
    subscription_cube.close_status.isNull()
).select(
    "industry_type",
    "total_revenue_gbp",
    "total_mrr_gbp",
    "subscription_count",
    "unique_customers"
).orderBy("total_revenue_gbp", ascending=False)

print("Total Revenue and Subscriptions by Industry")
print("="*50)
display(industry_kpi)